# Distributed Agent System Test Notebook

This notebook tests the distributed proxy/dispatcher architecture with multiple service agent clusters.

## Prerequisites
Before running this notebook, start the distributed services:
```bash
docker-compose -f docker-compose.distributed.yml up -d
```

## Test Scenarios
1. Proxy Service Health
2. Service Discovery
3. Direct Service Communication
4. Cross-Service Agent Communication
5. Health Check and Fault Tolerance
6. Multi-Service Collaborative Analysis


In [ ]:
# Setup and Configuration
import requests
import json
import time

# Service URLs
PROXY_URL = "http://localhost:8000"
SERVICE_A_URL = "http://localhost:8001"  # payment-service
SERVICE_B_URL = "http://localhost:8002"  # order-service

def pretty_print(data):
    """Pretty print JSON data."""
    print(json.dumps(data, indent=2, default=str))

print("Configuration loaded!")
print(f"Proxy URL: {PROXY_URL}")
print(f"Service A URL: {SERVICE_A_URL}")
print(f"Service B URL: {SERVICE_B_URL}")


In [ ]:
# Test 1: Proxy Service Health Check
print("=" * 60)
print("TEST 1: Proxy Service Health")
print("=" * 60)

try:
    response = requests.get(f"{PROXY_URL}/health", timeout=10)
    print(f"\nStatus Code: {response.status_code}")
    print("\nProxy Health Response:")
    pretty_print(response.json())
    
    if response.status_code == 200:
        print("\n✅ Proxy service is healthy!")
    else:
        print("\n❌ Proxy service returned non-200 status")
        
except requests.exceptions.ConnectionError:
    print("\n❌ Cannot connect to proxy service")
    print("Make sure you've started the services with:")
    print("docker-compose -f docker-compose.distributed.yml up -d")


In [ ]:
# Test 2: Service Discovery
print("=" * 60)
print("TEST 2: Service Discovery")
print("=" * 60)

# List all services
print("\n--- All Registered Services ---")
try:
    response = requests.get(f"{PROXY_URL}/services", timeout=10)
    print(f"Status Code: {response.status_code}")
    data = response.json()
    pretty_print(data)
    print(f"\nTotal services: {data.get('count', 0)}")
except Exception as e:
    print(f"Error: {e}")

# Discover by capability
print("\n--- Discover Services with 'payment' capability ---")
try:
    response = requests.get(f"{PROXY_URL}/discover", params={"capability": "payment"}, timeout=10)
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")

print("\n--- Discover Services with 'order' capability ---")
try:
    response = requests.get(f"{PROXY_URL}/discover", params={"capability": "order"}, timeout=10)
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")


In [ ]:
# Test 3: Direct Service Communication
print("=" * 60)
print("TEST 3: Direct Service Communication")
print("=" * 60)

# Check Service A health and info
print("\n--- Service A (payment-service) Health ---")
try:
    response = requests.get(f"{SERVICE_A_URL}/health", timeout=10)
    print(f"Status: {response.status_code}")
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")

print("\n--- Service A Info ---")
try:
    response = requests.get(f"{SERVICE_A_URL}/api/service-info", timeout=10)
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")

# Check Service B health and info
print("\n--- Service B (order-service) Health ---")
try:
    response = requests.get(f"{SERVICE_B_URL}/health", timeout=10)
    print(f"Status: {response.status_code}")
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")

print("\n--- Service B Info ---")
try:
    response = requests.get(f"{SERVICE_B_URL}/api/service-info", timeout=10)
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")


In [ ]:
# Test 4: Cross-Service Agent Communication
print("=" * 60)
print("TEST 4: Cross-Service Agent Communication")
print("=" * 60)

# Service A discovers other services
print("\n--- Service A Discovers Other Services ---")
try:
    response = requests.get(f"{SERVICE_A_URL}/api/discover-services", timeout=10)
    print("Services discovered by Service A:")
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")

# Service A queries Service B
print("\n--- Service A Queries Service B for Analysis ---")
test_error = """
2025-01-14 10:30:15,123 [ERROR] [payment-service]
Payment processing failed for order #12345
java.sql.SQLException: Connection timeout to payment gateway
  at com.payment.Gateway.connect(Gateway.java:87)
  at com.payment.Processor.processPayment(Processor.java:145)
"""

try:
    response = requests.post(
        f"{SERVICE_A_URL}/api/query-service/order-service",
        json={"query": test_error},
        timeout=120
    )
    print(f"Status: {response.status_code}")
    result = response.json()
    pretty_print(result)
except Exception as e:
    print(f"Error: {e}")


In [ ]:
# Test 5: Multi-Service Collaborative Analysis
print("=" * 60)
print("TEST 5: Multi-Service Collaborative Analysis")
print("=" * 60)

# Complex error that spans multiple services
complex_error = """
2025-01-14 12:15:42,123 [ERROR] [payment-service]
Critical payment processing failure detected.

Transaction Details:
- Order ID: #98765
- Amount: $1,250.00
- Customer: customer_456

Error Stack:
javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException
Unable to acquire JDBC Connection
  at org.hibernate.internal.ExceptionConverterImpl.convert(ExceptionConverterImpl.java:154)
Caused by: java.sql.SQLTransientConnectionException: HikariPool-1 - Connection timeout after 30000ms.

This error may be affecting order fulfillment and inventory management.
"""

print("\nSubmitting complex error for multi-service analysis...")
print("\n" + "-" * 40)
print("Error being analyzed:")
print(complex_error[:400] + "...")
print("-" * 40)

# Query Service A for comprehensive analysis
print("\n--- Querying payment-service for comprehensive analysis ---")
print("(This may take a minute as the agent queries multiple services)")

try:
    start_time = time.time()
    response = requests.post(
        f"{SERVICE_A_URL}/api/query",
        json={
            "query": f"""
Analyze this critical payment error and provide a comprehensive report.
Use cross-service analysis to understand the full impact.

Error Log:
{complex_error}
"""
        },
        timeout=180
    )
    duration = time.time() - start_time
    
    print(f"\nStatus: {response.status_code}")
    print(f"Duration: {duration:.2f} seconds")
    
    if response.status_code == 200:
        result = response.json()
        print("\n" + "=" * 60)
        print("COMPREHENSIVE ANALYSIS REPORT")
        print("=" * 60)
        print(result.get('reply', 'No reply')[:3000])
    else:
        print(f"Error: {response.text}")
        
except Exception as e:
    print(f"Error: {e}")


In [ ]:
# Test 6: Summary and System Status
print("=" * 60)
print("DISTRIBUTED SYSTEM TEST SUMMARY")
print("=" * 60)

tests_passed = 0
tests_failed = 0

# Check proxy
try:
    r = requests.get(f"{PROXY_URL}/health", timeout=5)
    if r.status_code == 200:
        print("✅ Proxy: Healthy")
        tests_passed += 1
    else:
        print("❌ Proxy: Unhealthy")
        tests_failed += 1
except:
    print("❌ Proxy: Unavailable")
    tests_failed += 1

# Check Service A
try:
    r = requests.get(f"{SERVICE_A_URL}/health", timeout=5)
    if r.status_code == 200:
        print("✅ Service A (payment-service): Healthy")
        tests_passed += 1
    else:
        print("❌ Service A: Unhealthy")
        tests_failed += 1
except:
    print("❌ Service A: Unavailable")
    tests_failed += 1

# Check Service B
try:
    r = requests.get(f"{SERVICE_B_URL}/health", timeout=5)
    if r.status_code == 200:
        print("✅ Service B (order-service): Healthy")
        tests_passed += 1
    else:
        print("❌ Service B: Unhealthy")
        tests_failed += 1
except:
    print("❌ Service B: Unavailable")
    tests_failed += 1

# Registered services
try:
    r = requests.get(f"{PROXY_URL}/services", timeout=5)
    data = r.json()
    print(f"\n📊 Registered Services: {data.get('count', 0)}")
    for svc in data.get('services', []):
        status_icon = "✅" if svc['status'] == 'healthy' else "⚠️"
        print(f"   {status_icon} {svc['name']}: {svc.get('capabilities', [])}")
except:
    pass

print(f"\n--- Results ---")
print(f"Tests Passed: {tests_passed}")
print(f"Tests Failed: {tests_failed}")

print("\n" + "=" * 60)
print("CLEANUP INSTRUCTIONS")
print("=" * 60)
print("""
To stop all distributed services:
  docker-compose -f docker-compose.distributed.yml down

To stop and remove all data:
  docker-compose -f docker-compose.distributed.yml down -v

To view logs:
  docker-compose -f docker-compose.distributed.yml logs -f
""")
